# Ames Housing — sanitizado para regresion / redes

Objetivo: `SalePrice` (regresion). Variables categoricas: one-hot; numericas: imputacion + estandarizacion.

In [25]:
import sys
from pathlib import Path

# Raiz del proyecto (donde esta .env y load_project_env.py)
def _project_root() -> Path:
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "load_project_env.py").exists():
            return cand
    return p

ROOT = _project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from load_project_env import load_env, data_path

load_env()
print("PROJECT_ROOT:", ROOT)


PROJECT_ROOT: C:\Users\Hp\OneDrive\Escritorio\primerparcial_ia


In [26]:
import os
import pandas as pd
import numpy as np

csv_path = data_path("DATA_AMES_TRAIN")
df = pd.read_csv(csv_path)
target_col = "SalePrice"
y = df[target_col].astype(float).values
X = df.drop(columns=[target_col])
# Quitar ID si existe
for c in ('Id', 'Order', 'PID'):
    if c in X.columns:
        X = X.drop(columns=[c])
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]
print(X.shape, len(num_cols), len(cat_cols))

(2197, 79) 36 43


In [27]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def build_preprocessor_regression(X, numeric, categorical):
    num = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    cat = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )
    return ColumnTransformer(
        transformers=[("num", num, numeric), ("cat", cat, categorical)],
        remainder="drop",
    )


In [28]:
prep = build_preprocessor_regression(X, num_cols, cat_cols)
X_matrix = prep.fit_transform(X)
print('Matriz lista:', X_matrix.shape, '| y:', y.shape)
# Listo para train_test_split, MLPRegressor, etc.

Matriz lista: (2197, 303) | y: (2197,)
